In [1]:
import os
import sys

module_path = os.path.abspath(os.path.join("../"))
if module_path not in sys.path:
    sys.path.append(module_path)

from config.loaders.dataset_config_loader import DatasetConfig
from datasets import load_dataset, load_from_disk
from utils.data_io import make_dir, save_jsonl
from utils.question_formatter import QuestionFormatter

In [2]:
dataset = "seed"
ds_config = DatasetConfig()
ds_paths = ds_config.get_ds_paths(dataset)
ds_paths

{'scene_understanding': {'imgs': 'data/datasets/seed/imgs/scene_understanding',
  'questions': 'data/datasets/seed/q_scene_understanding.jsonl'},
 'visual_reasoning': {'imgs': 'data/datasets/seed/imgs/visual_reasoning',
  'questions': 'data/datasets/seed/q_visual_reasoning.jsonl'}}

In [3]:
question_formatter = QuestionFormatter(dataset)

### Downloading and extracting data (takes time)

Skip to next section if arrow files are already present

In [ ]:
ds = load_dataset("lmms-lab/SEED-Bench-2", split="test")

In [4]:
ds[0]

{'answer': 'A',
 'choice_a': 'One',
 'choice_b': 'Two',
 'choice_c': 'Three',
 'choice_d': 'Four',
 'data_id': ['1454426_2591111986'],
 'data_type': 'Single Image',
 'data_source': 'cc3m',
 'level': 'L1',
 'question': 'How many towels are in the image?',
 'question_id': '5_0',
 'question_type_id': 5,
 'image': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1280x853>],
 'subpart': 'Single-Image & Text Comprehension',
 'version': 'v1'}

In [5]:
# This comes from the repository of SEED-Bench-2
qs_type_ids_p1_1 = {
    "Scene Understanding": 1,
    "Instance Identity": 2,
    "Instance Attributes": 3,
    "Instance Location": 4,
    "Instances Counting": 5,
    "Spatial Relation": 6,
    "Instance Interaction": 7,
    "Visual Reasoning": 8,
    "Text Understanding": 9,
    "Celebrity Recognition": 10,
    "Landmark Recognition": 11,
    "Chart Understanding": 12,
    "Visual Referring Expression": 13,
    "Science Knowledge": 14,
    "Emotion Recognition": 15,
    "Visual Mathematics": 16,
}

qs_type_ids_p1_2 = {"Difference Spotting": 17, "Meme Comprehension": 18}

qs_type_ids_p1_3 = {
    "Global Video Understanding": 19,
    "Action Recognition": 20,
    "Action Prediction": 21,
    "Procedure Understanding": 22,
}

qs_type_ids_p2 = {"In-Context Captioning": 23, "Interleaved Image-Text Analysis": 24}

qs_type_ids_p3 = {
    "Text-to-Image Generation": 25,
    "Next Image Prediction": 26,
    "Text-Image Creation": 27,
}

Chunk dataset

In [6]:
cat_to_keep = {
    "Scene Understanding": 1,
    "Visual Reasoning": 8,
}

In [ ]:
seed_p1_scene_understanding = ds.filter(
    lambda entry: entry["question_type_id"] == 1, num_proc=2
)
print(len(seed_p1_scene_understanding))

In [ ]:
seed_p1_visual_reasoning = ds.filter(
    lambda entry: entry["question_type_id"] == 8, num_proc=2
)
print(len(seed_p1_visual_reasoning))

Backup filtered data, if needed afterwards

In [10]:
# Scene understanding
seed_p1_scene_understanding.save_to_disk(
    "./data_backup/scene_understanding", max_shard_size="40MB"
)

# Visual reasoning
seed_p1_visual_reasoning.save_to_disk("./data_backup/visual_reasoning")

Saving the dataset (0/11 shards):   0%|          | 0/3158 [00:00<?, ? examples/s]

### Scene Understanding
Loading the pre-processed scene understanding split from disk

In [4]:
c = "scene_understanding"
backup_path = f"../data/datasets/seed/data_backup/{c}"
seed_su = load_from_disk(backup_path)
seed_su

Loading dataset from disk:   0%|          | 0/136 [00:00<?, ?it/s]

Dataset({
    features: ['answer', 'choice_a', 'choice_b', 'choice_c', 'choice_d', 'data_id', 'data_type', 'data_source', 'level', 'question', 'question_id', 'question_type_id', 'image', 'subpart', 'version'],
    num_rows: 3158
})

In [5]:
seed_su[4]

{'answer': 'C',
 'choice_a': 'The gift bag and candy canes in the foreground.',
 'choice_b': 'The full moon in the background.',
 'choice_c': 'The character on a plane with a bag of presents.',
 'choice_d': 'The red heart with white letters.',
 'data_id': ['3201538_4179577514'],
 'data_type': 'Single Image',
 'data_source': 'cc3m',
 'level': 'L1',
 'question': 'What is the focus of the image?',
 'question_id': '1_4',
 'question_type_id': 1,
 'image': [<PIL.Image.Image image mode=RGB size=1300x900>],
 'subpart': 'Single-Image & Text Comprehension',
 'version': 'v1'}

In [6]:
q_su = []

img_path_col_su = [""] * len(seed_su)
if "img_path" in seed_su:
    seed_su = seed_su.remove_columns("img_path")

seed_su = seed_su.add_column("img_path", img_path_col_su)

out_dir = f"../{ds_paths[c]['imgs']}"
make_dir(out_dir)

# Scene Understanding
for entry in seed_su:
    img_path = f"{len(q_su)}.jpg"
    entry["img_path"] = f"{out_dir}/{img_path}"

    curr_img = entry["image"][0].convert("RGB")
    curr_img.save(entry["img_path"])
    entry.pop("image")

    q_su.append(entry)

[Errno 17] File exists: '../data/datasets/seed/imgs/scene_understanding'


Save question data

In [7]:
new_q_su = [question_formatter.format(raw_q, c) for raw_q in q_su]
save_jsonl(new_q_su, f"../{ds_paths[c]['questions']}")

### Visual Reasoning
Loading the pre-processed visual reasoning split from disk

In [8]:
c = "visual_reasoning"
backup_path = f"../data/datasets/seed/data_backup/{c}"
seed_vr = load_from_disk(backup_path)
seed_vr

Dataset({
    features: ['answer', 'choice_a', 'choice_b', 'choice_c', 'choice_d', 'data_id', 'data_type', 'data_source', 'level', 'question', 'question_id', 'question_type_id', 'image', 'subpart', 'version'],
    num_rows: 331
})

In [9]:
seed_vr[0]

{'answer': 'D',
 'choice_a': 'Guitar',
 'choice_b': 'Microphone',
 'choice_c': 'Arm',
 'choice_d': 'Shoe',
 'data_id': ['2871019_2604587489'],
 'data_type': 'Single Image',
 'data_source': 'cc3m',
 'level': 'L1',
 'question': 'Which object in the image has an attribute of "leather"?',
 'question_id': '8_0',
 'question_type_id': 8,
 'image': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1365x2048>],
 'subpart': 'Single-Image & Text Comprehension',
 'version': 'v1'}

In [10]:
q_vr = []

img_path_col_vr = [""] * len(seed_vr)
if "img_path" in seed_vr:
    seed_vr = seed_vr.remove_columns("img_path")

seed_vr = seed_vr.add_column("img_path", img_path_col_vr)

out_dir = f"../{ds_paths[c]['imgs']}"
make_dir(out_dir)

# Visual Reasoning
for entry in seed_vr:
    img_path = f"{len(q_vr)}.jpg"
    entry["img_path"] = f"{out_dir}/{img_path}"

    curr_img = entry["image"][0].convert("RGB")
    curr_img.save(entry["img_path"])
    entry.pop("image")

    q_vr.append(entry)

[Errno 17] File exists: '../data/datasets/seed/imgs/visual_reasoning'


In [11]:
new_q_vr = [question_formatter.format(raw_q, c) for raw_q in q_vr]
save_jsonl(new_q_vr, f"../{ds_paths[c]['questions']}")